In [1]:
import pandas as pd

df = pd.read_csv('Online Retail.csv')

print("Data loaded successfully. Shape:", df.shape)
df.head()

Data loaded successfully. Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [2]:
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df = df.dropna(subset=['CustomerID'])
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print("Data cleaned. Shape:", df.shape)
df.head()

Data cleaned. Shape: (397884, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [4]:
import datetime as dt

today = df['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (today - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

r_bins, r_edges = pd.qcut(rfm['Recency'], 5, retbins=True, duplicates='drop')
f_bins, f_edges = pd.qcut(rfm['Frequency'], 4, retbins=True, duplicates='drop')
m_bins, m_edges = pd.qcut(rfm['Monetary'], 5, retbins=True, duplicates='drop')

rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=list(range(len(r_edges)-1, 0, -1)), duplicates='drop')
rfm['F_Score'] = pd.qcut(rfm['Frequency'], 4, labels=list(range(1, len(f_edges))), duplicates='drop')
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=list(range(1, len(m_edges))), duplicates='drop')

rfm['RFM_Score'] = rfm['R_Score'].astype(int) + rfm['F_Score'].astype(int) + rfm['M_Score'].astype(int)

def segment_customer(score):
    if score >= 13:
        return 'High Value Loyal Customers'
    elif score >= 10:
        return 'Potential Customers'
    elif score >= 7:
        return 'General Customers'
    else:
        return 'Price Sensitive / New Customers'

rfm['Customer Segment'] = rfm['RFM_Score'].apply(segment_customer)

print(rfm['Customer Segment'].value_counts())
rfm.head()

Customer Segment
Price Sensitive / New Customers    1766
General Customers                  1247
Potential Customers                 977
High Value Loyal Customers          348
Name: count, dtype: int64


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Customer Segment
0,12346.0,326,1,77183.60,1,1,5,7,General Customers
1,12347.0,2,7,4310.00,5,3,5,13,High Value Loyal Customers
2,12348.0,75,4,1797.24,2,2,4,8,General Customers
3,12349.0,19,1,1757.55,4,1,4,9,General Customers
4,12350.0,310,1,334.40,1,1,2,4,Price Sensitive / New Customers


In [5]:
df = df.merge(rfm[['CustomerID', 'Customer Segment']], on='CustomerID', how='left')

bundle = df.groupby(['InvoiceNo', 'Customer Segment'])['Description'].agg(list).reset_index()
bundle_amount = df.groupby(['InvoiceNo', 'Customer Segment'])['TotalAmount'].sum().reset_index()
bundle = bundle.merge(bundle_amount, on=['InvoiceNo', 'Customer Segment'])
bundle = bundle.explode('Description')

summary = []
for seg in rfm['Customer Segment'].unique():
    seg_data = bundle[bundle['Customer Segment'] == seg]
    top5 = seg_data.groupby('Description')['TotalAmount'].agg(['count', 'mean']).sort_values('count', ascending=False).head(5)
    top5 = top5.reset_index()
    top5['Customer Segment'] = seg
    summary.append(top5)

final_summary = pd.concat(summary, ignore_index=True)
final_summary = final_summary[['Customer Segment', 'Description', 'count', 'mean']]
final_summary.columns = ['Customer Segment', 'Recommended Bundle', 'Frequency', 'Average Payment Amount']

final_summary.to_csv('recommendations.csv', index=False)
print("Recommendations saved to recommendations.csv")
final_summary

Recommendations saved to recommendations.csv


,Customer Segment,Recommended Bundle,Frequency,Average Payment Amount
0,General Customers,WHITE HANGING HEART T-LIGHT HOLDER,401,484.547631
1,General Customers,REGENCY CAKESTAND 3 TIER,363,550.296752
2,General Customers,ASSORTED COLOUR BIRD ORNAMENT,309,428.166735
3,General Customers,PARTY BUNTING,268,466.291045
4,General Customers,REX CASH+CARRY JUMBO SHOPPER,241,318.252739
5,High Value Loyal Customers,JUMBO BAG RED RETROSPOT,722,950.294501
6,High Value Loyal Customers,WHITE HANGING HEART T-LIGHT HOLDER,665,806.776992
7,High Value Loyal Customers,REGENCY CAKESTAND 3 TIER,572,1179.212343
8,High Value Loyal Customers,LUNCH BAG RED RETROSPOT,564,980.895443
9,High Value Loyal Customers,PARTY BUNTING,482,740.051577


In [6]:
print("=== Final Summary ===")
print("Number of customer segments:", len(final_summary['Customer Segment'].unique()))
print("\nTop recommendations per segment:")
print(final_summary.groupby('Customer Segment').head(5))
print("\nRecommendations file saved as 'recommendations.csv'")
print("Ready for Streamlit app.")

=== Final Summary ===
Number of customer segments: 4

Top recommendations per segment:
                   Customer Segment                  Recommended Bundle  \
0                 General Customers  WHITE HANGING HEART T-LIGHT HOLDER   
1                 General Customers            REGENCY CAKESTAND 3 TIER   
2                 General Customers       ASSORTED COLOUR BIRD ORNAMENT   
3                 General Customers                       PARTY BUNTING   
4                 General Customers        REX CASH+CARRY JUMBO SHOPPER   
5        High Value Loyal Customers             JUMBO BAG RED RETROSPOT   
6        High Value Loyal Customers  WHITE HANGING HEART T-LIGHT HOLDER   
7        High Value Loyal Customers            REGENCY CAKESTAND 3 TIER   
8        High Value Loyal Customers             LUNCH BAG RED RETROSPOT   
9        High Value Loyal Customers                       PARTY BUNTING   
10  Price Sensitive / New Customers  WHITE HANGING HEART T-LIGHT HOLDER   
11  Price Sen